In [1]:
# có 1 khâu tiền xử lý input trước khi RAG
# input gồm có text và image(optional). Lúc này mong muốn là dùng 1 LLM (local nhỏ thôi) trích ra dữ kiện diễn đạt lại thành những fact nhỏ cần kiểm chứng.
# Dùng 1 model j đó mô tả được hình ảnh và liên kết nó với text. (dùng VLM + fewshot prompting đừng bắt nó làm việc quá phức tạp)
# Dùng kỹ thuật Chain of Thoughts để bắt LLM làm tác vụ. 
# Chú ý là các model local này nhỏ nên chia tác vụ riêng ra mỗi lần prompt rồi dùng kết quả lần trước cho lần sau.

In [2]:
# Xử lý Hình ảnh: Qwen2-VL-2B-Instruct hoặc Llama-3.2-11B-, Kỹ thuật áp dụng: Few-shot Prompting
# Trích xuất Sự kiện: Qwen2.5-3B-Instruct hoặc Llama-3.2-3B-Instruct,  Kỹ thuật áp dụng: Chain of Thought (CoT).
# Chaining Tasks: Ollama


In [3]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils pillow requests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 53.1 MB/s eta 0:00:00


In [4]:
import torch
import json
import os
import re
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Đang tải mô hình Qwen2-VL-7B-Instruct (Float16)...")
model_id = "Qwen/Qwen2-VL-7B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 # Chạy nguyên bản, không dùng 4-bit
)
processor = AutoProcessor.from_pretrained(model_id)

def run_qwen_inference(messages, max_tokens=2048):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.4, do_sample=True, repetition_penalty=1.1)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return output_text[0]

Đang tải mô hình Qwen2-VL-7B-Instruct (Float16)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

chat_template.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# # =====================================================================
# # INPUT TỪ USER
# # =====================================================================
# user_claim = "Nguyễn Long Vũ đã bị Cơ quan Cảnh sát điều tra CATP Hà Nội ra quyết định tạm giữ hình sự về hành vi gây rối trật tự công cộng."
# # Thay đổi đường dẫn ảnh thực tế trên Kaggle của bạn
# user_image_path = "/kaggle/input/datasets/lctrnhu/demodataset-nlp/demodata/post_170_img_0.jpg" 

# print(f"--- BẮT ĐẦU XỬ LÝ CLAIM: {user_claim[:50]}... ---\n")

# # =====================================================================
# # TÁC VỤ 1: VLM - IMAGE-TEXT RELATIONSHIP (PROMPT TIẾNG ANH + FEW-SHOT)
# # =====================================================================
# image_context = "No image provided."
# if user_image_path and os.path.exists(user_image_path):
#     print("[Step 1/3] Analyzing Image-Text Relationship...")
#     prompt_vlm = f"""Task: Analyze the relevance between the image and the claim.
    
#     [EXAMPLE]
#     Claim: "Hanoi is flooded after the storm."
#     Image: [Photo of cars in deep water]
#     Output: The image is HIGHLY RELEVANT. It depicts a flooded street, which directly supports the context of the claim.

#     [YOUR TASK]
#     Claim: "{user_claim}"
#     Instruction: Describe what you see and how it relates to the claim in Vietnamese.
#     Output:"""
    
#     msg_1 = [{"role": "user", "content": [{"type": "image", "image": f"file://{user_image_path}"}, {"type": "text", "text": prompt_vlm}]}]
#     image_context = run_qwen_inference(msg_1)
#     print(f"VLM Output: {image_context[:100]}...")

# # =====================================================================
# # TÁC VỤ 2: LLM - ATOMIC FACT EXTRACTION (CoT + FEW-SHOT)
# # =====================================================================
# print("[Step 2/3] Extracting Atomic Facts...")
# prompt_facts = f"""Task: Breakdown the claim into verifiable atomic facts.
# Rules: Each fact must be a single, complete sentence in Vietnamese.

# [EXAMPLE]
# Claim: "At 10 PM yesterday, John stole a phone at the park and ran away."
# Thinking: The claim has time (10 PM), subject (John), action 1 (stole phone), location (park), action 2 (ran away).
# Facts:
# 1. Sự việc diễn ra vào lúc 10 giờ tối ngày hôm qua.
# 2. Đối tượng tên là John.
# 3. John đã thực hiện hành vi trộm điện thoại.
# 4. Sự việc xảy ra tại khu vực công viên.

# [YOUR TASK]
# Claim: "{user_claim}"
# Thinking:"""

# msg_2 = [{"role": "user", "content": [{"type": "text", "text": prompt_facts}]}]
# facts_output = run_qwen_inference(msg_2)
# # Tách lấy phần sau chữ "Facts:"
# atomic_facts = facts_output.split("Facts:")[-1].strip() if "Facts:" in facts_output else facts_output

# # =====================================================================
# # TÁC VỤ 3: LLM - HyDE & JSON GENERATION
# # =====================================================================
# print("[Step 3/3] Generating HyDE and Final JSON for RAG...")
# prompt_hyde = f"""Task: Create a Preprocessing JSON for a RAG system. 
# Apply HyDE (Hypothetical Document Embeddings): write a short, hypothetical news report in Vietnamese that would perfectly verify these facts.

# [INPUT]
# Image Context: {image_context}
# Facts: {atomic_facts}

# [OUTPUT FORMAT]
# Return ONLY a JSON object. NO markdown, NO explanation.
# {{
#   "is_multimodal": true,
#   "image_analysis": "Mô tả ngắn gọn sự liên quan của ảnh bằng tiếng Việt",
#   "queries": ["query 1", "query 2"],
#   "hyde_doc": "Một đoạn văn bản giả định (3-4 câu) phong cách báo chí hoặc báo cáo điều tra chứa đầy đủ các sự kiện trên."
# }}

# [YOUR TASK]
# Output JSON:"""

# msg_3 = [{"role": "user", "content": [{"type": "text", "text": prompt_hyde}]}]
# final_output = run_qwen_inference(msg_3)

# # =====================================================================
# # KẾT QUẢ CUỐI CÙNG
# # =====================================================================
# print("\n" + "="*30 + " FINAL PREPROCESSED JSON " + "="*30)
# # Dùng Regex để đảm bảo chỉ lấy đúng khối JSON
# json_match = re.search(r'\{.*\}', final_output, re.DOTALL)
# if json_match:
#     print(json_match.group(0))
# else:
#     print(final_output)

In [6]:
# =====================================================================
# INPUT TỪ USER
# =====================================================================
# user_claim = "Nguyễn Long Vũ đã bị Cơ quan Cảnh sát điều tra CATP Hà Nội ra quyết định tạm giữ hình sự về hành vi gây rối trật tự công cộng."
user_claim = "Giữa hội trường, ông Tạ Hoàng yêu cầu mọi người khẩn trương thi hành các nhiệm vụ còn dang dở."
# Thay đổi đường dẫn ảnh thực tế trên Kaggle của bạn
user_image_path = "/kaggle/input/datasets/lctrnhu/anhnestestthu/post_3_img_0.jpg" 

print(f"--- BẮT ĐẦU XỬ LÝ CLAIM: {user_claim[:50]}... ---\n")

# =====================================================================
# TÁC VỤ 1: VLM - IMAGE-TEXT RELATIONSHIP
# =====================================================================
image_context = "No image provided."
has_image = False

if user_image_path and os.path.exists(user_image_path):
    has_image = True
    print("[Step 1/3] Analyzing Image-Text Relationship...")
    prompt_vlm = f"""Task: Describe EXACTLY what you see in the provided image. 
    Focus on objective physical details: people, clothing, actions, text, and surroundings.
    CRITICAL: Output a simple bulleted list in Vietnamese. DO NOT write a continuous paragraph.
    
    [YOUR TASK]
    Output (Danh sách gạch đầu dòng các sự kiện thấy trong ảnh):"""
    
    msg_1 = [{"role": "user", "content": [{"type": "image", "image": f"file://{user_image_path}"}, {"type": "text", "text": prompt_vlm}]}]
    image_context = run_qwen_inference(msg_1)
    print(f"VLM Output:\n{image_context[:200]}...")
else:
    print("[Step 1/3] Không tìm thấy ảnh. Bỏ qua VLM.")

# =====================================================================
# TÁC VỤ 2: LLM - ATOMIC FACT EXTRACTION (CoT + FEW-SHOT)
# =====================================================================
print("[Step 2/3] Extracting Atomic Facts...")
prompt_facts = f"""Task: Breakdown the claim into verifiable atomic facts.
Rules: Each fact must be a single, complete sentence in Vietnamese.

[EXAMPLE]
Claim: "At 10 PM yesterday, John stole a phone at the park and ran away."
Thinking: The claim has time (10 PM), subject (John), action 1 (stole phone), location (park), action 2 (ran away).
Facts:
1. Sự việc diễn ra vào lúc 10 giờ tối ngày hôm qua.
2. Đối tượng tên là John.
3. John đã thực hiện hành vi trộm điện thoại.
4. Sự việc xảy ra tại khu vực công viên.

[YOUR TASK]
Claim: "{user_claim}"
Thinking:"""

msg_2 = [{"role": "user", "content": [{"type": "text", "text": prompt_facts}]}]
facts_output = run_qwen_inference(msg_2)
atomic_facts = facts_output.split("Facts:")[-1].strip() if "Facts:" in facts_output else facts_output

# =====================================================================
# TÁC VỤ 3: LLM - TỔNG HỢP JSON ĐÚNG FORMAT YÊU CẦU
# =====================================================================
print("[Step 3/3] Generating HyDE and Final JSON for RAG...")

# Chuyển boolean Python sang string "true"/"false" cho JSON
img_provided_str = "true" if has_image else "false"

prompt_hyde = f"""Task: Create a Preprocessing JSON for a RAG system based ONLY on the [INPUT].
CRITICAL RULE: Do not mix the inputs. Map them exactly as instructed below.

[INPUT]
Claim: "{user_claim}"
Image Context: "{image_context}"
Text Facts: "{atomic_facts}"

[INSTRUCTIONS FOR JSON FIELDS]
1. "image_facts": Lấy các gạch đầu dòng từ 'Image Context' bỏ vào đây.
2. "text_facts": Lấy các câu từ 'Text Facts' bỏ vào đây. Không được lấy từ ảnh.
3. "normalized_facts": Gộp mảng 1 và mảng 2 lại.
4. "alignment": BẮT BUỘC chỉ được mở đầu bằng 1 trong 3 cụm từ: "Khớp hoàn toàn", "Không khớp", hoặc "Khớp một phần". Sau đó giải thích ngắn gọn trong đúng 1 câu. (Ví dụ: "Khớp một phần. Ảnh có cảnh hội trường và người mặc quân phục nhưng không thể xác định ai là ông Tạ Hoàng").
5. "rag_queries": Tạo 3 từ khóa tìm kiếm (mỗi từ khóa 2-5 chữ) chỉ dựa vào 'Claim'.
6. "hyde_doc": Viết 2 đoạn văn giả định. ĐỂ CHỐNG LẶP: Đoạn 1 viết theo phong cách "Bản tin thời sự". Đoạn 2 viết theo phong cách "Báo cáo nội bộ ngắn gọn". CHỈ dùng thông tin từ Claim, lờ đi hoàn toàn chi tiết hình ảnh.

[OUTPUT FORMAT]
Return ONLY a valid JSON object. Fill the empty arrays [] with actual string values:
{{
  "claim": "{user_claim}",
  "image_provided": {img_provided_str},
  "image_facts": [],
  "text_facts": [],
  "normalized_facts": [],
  "alignment": [""],
  "rag_queries": [],
  "hyde_doc": []
}}

[YOUR TASK]
Output JSON:"""

msg_3 = [{"role": "user", "content": [{"type": "text", "text": prompt_hyde}]}]
final_output = run_qwen_inference(msg_3)

# =====================================================================
# KẾT QUẢ CUỐI CÙNG
# =====================================================================
print("\n" + "="*30 + " FINAL PREPROCESSED JSON " + "="*30)
# Dùng Regex để đảm bảo chỉ lấy đúng khối JSON
json_match = re.search(r'\{.*\}', final_output, re.DOTALL)
if json_match:
    print(json_match.group(0))
else:
    print(final_output)

--- BẮT ĐẦU XỬ LÝ CLAIM: Giữa hội trường, ông Tạ Hoàng yêu cầu mọi người kh... ---

[Step 1/3] Analyzing Image-Text Relationship...
VLM Output:
- Một nhóm người gồm bác sĩ và nhân viên y tế đang đứng trong phòng bệnh viện.
- Người phụ nữ ở giữa mặc áo xanh lá cây và đội mũ bảo hộ màu xanh dương.
- Người đàn ông bên phải mặc áo trắng, đang cầm...
[Step 2/3] Extracting Atomic Facts...
[Step 3/3] Generating HyDE and Final JSON for RAG...

============================== FINAL PREPROCESSED JSON ==============================
{
  "claim": "Giữa hội trường, ông Tạ Hoàng yêu cầu mọi người khẩn trương thi hành các nhiệm vụ còn dang dở.",
  "image_provided": true,
  "image_facts": [
    "Nhóm người gồm bác sĩ và nhân viên y tế",
    "Người phụ nữ mặc áo xanh lá cây và đội mũ bảo hộ màu xanh dương",
    "Người đàn ông mặc áo trắng cầm hộp quà tặng",
    "Hộp quà tặng có chữ 'ĐỀNH VIỆN TƯ ĐỜI'",
    "Thẻ giấy trên tay người đàn ông có chữ 'HELLO 2026'",
    "Bé gái nằm trên giường bệnh, phủ kín bở

In [7]:
# =====================================================================
# KẾT QUẢ CUỐI CÙNG VÀ LƯU FILE
# =====================================================================
print("\n" + "="*30 + " FINAL PREPROCESSED JSON " + "="*30)
# Dùng Regex để đảm bảo chỉ lấy đúng khối JSON
json_match = re.search(r'\{.*\}', final_output, re.DOTALL)

if json_match:
    json_str = json_match.group(0)
    print(json_str)
    
    # ---------------------------------------------------------
    # THÊM MỚI: LƯU JSON VÀO KAGGLE WORKING DIRECTORY
    # ---------------------------------------------------------
    output_filename = "preprocessed_claim.json"
    output_path = os.path.join("/kaggle/working", output_filename)
    
    try:
        # Parse lại chuỗi JSON để đảm bảo format chuẩn xác và thụt lề đẹp (indent=4)
        parsed_json = json.loads(json_str)
        
        # Mở file với chế độ 'w' (write) và encoding utf-8 để không bị lỗi font tiếng Việt
        with open(output_path, "w", encoding="utf-8") as json_file:
            json.dump(parsed_json, json_file, ensure_ascii=False, indent=4)
            
        print("\n" + "="*50)
        print(f"[THÀNH CÔNG] Đã lưu dữ liệu định dạng JSON tại: {output_path}")
        print("Bạn có thể tải file này ở panel 'Output' bên phải màn hình Kaggle.")
        print("="*50)
        
    except json.JSONDecodeError:
        print("\n[LỖI] Chuỗi trích xuất được không phải là JSON hợp lệ. Không thể lưu file.")
    except Exception as e:
        print(f"\n[LỖI HỆ THỐNG] Quá trình lưu file gặp sự cố: {e}")

else:
    print(final_output)
    print("\n[LỖI] Không tìm thấy khối JSON trong output của mô hình.")


============================== FINAL PREPROCESSED JSON ==============================
{
  "claim": "Giữa hội trường, ông Tạ Hoàng yêu cầu mọi người khẩn trương thi hành các nhiệm vụ còn dang dở.",
  "image_provided": true,
  "image_facts": [
    "Nhóm người gồm bác sĩ và nhân viên y tế",
    "Người phụ nữ mặc áo xanh lá cây và đội mũ bảo hộ màu xanh dương",
    "Người đàn ông mặc áo trắng cầm hộp quà tặng",
    "Hộp quà tặng có chữ 'ĐỀNH VIỆN TƯ ĐỜI'",
    "Thẻ giấy trên tay người đàn ông có chữ 'HELLO 2026'",
    "Bé gái nằm trên giường bệnh, phủ kín bởi khăn xanh dương",
    "Có nhiều người khác đang chăm sóc hoặc thăm hỏi cô ấy"
  ],
  "text_facts": [
    "Sự kiện xảy ra giữa hội trường",
    "Người nói là ông Tạ Hoàng",
    "Ông Tạ Hoàng đã yêu cầu mọi người",
    "Ông Tạ Hoàng yêu cầu mọi người khẩn trương thực hiện các nhiệm vụ còn dang dở"
  ],
  "normalized_facts": [
    "Sự kiện xảy ra giữa hội trường",
    "Người nói là ông Tạ Hoàng",
    "Ông Tạ Hoàng đã yêu cầu mọi người",